# **Lezione 2.4 - Api in flask**

v1 semplice

In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)

todo = [
    "Comprare il pane",
    "Lavare la macchina",
    "Comprare il latte"
]

@app.route("/todos", methods=["GET", "POST"])
def all_todos():
    if request.method == "GET":
        return jsonify(todo)
    if request.method == "POST":
        if not request.is_json or "task" not in request.json:
            return "Parametro Task richiesto", 400
        todo.append(request.json["task"])
        return f"Task numero {len(todo)} creato", 201

@app.route("/todos/<int:todo_id>", methods=["GET", "PUT", "DELETE"])
def single_todo_operation(todo_id):
    try:
       todo[todo_id]
    except:
        return "Id todo non trovato", 404
    if request.method == "GET":
        return jsonify(todo[todo_id])
    elif request.method == "PUT":
        if not request.is_json or "task" not in request.json:
            return "Parametro task richiesto", 400
        todo[todo_id] = request.json["task"]
        return f"Todo {todo_id} aggiornato"
    elif request.method == "DELETE":
        todo.pop(todo_id)
        return f"Todo {todo_id} eliminato", 200

if __name__ == "__main__":
app.run(debug=True)

v2 complessa


In [ ]:
from flask import Flask, request, jsonify
import requests

app = Flask(__name__)

# Lista di compiti come semplice database in memoria
todos = [
    {"id": 1, "task": "Fare la spesa"},
    {"id": 2, "task": "Studiare Flask"}
]

# GET /todos - Ottieni tutti i compiti
@app.route('/todos', methods=['GET'])
def get_todos():
    return jsonify(todos)

# GET /todos/<id> - Ottieni un compito specifico
@app.route('/todos/<int:todo_id>', methods=['GET'])
def get_todo(todo_id):
    todo = next((todo for todo in todos if todo['id'] == todo_id), None)
    if todo:
        return jsonify(todo)
    return jsonify({"error": "Compito non trovato"}), 404

# POST /todos - Aggiungi un nuovo compito
@app.route('/todos', methods=['POST'])
def add_todo():
    if not request.json or 'task' not in request.json:
        return jsonify({"error": "Il campo 'task' è obbligatorio"}), 400
    new_todo = {
        'id': todos[-1]['id'] + 1 if todos else 1,
        'task': request.json['task']
    }
    todos.append(new_todo)
    return jsonify(new_todo), 201

# PUT /todos/<id> - Aggiorna un compito esistente
@app.route('/todos/<int:todo_id>', methods=['PUT'])
def update_todo(todo_id):
    todo = next((todo for todo in todos if todo['id'] == todo_id), None)
    if not todo:
        return jsonify({"error": "Compito non trovato"}), 404
    if not request.json or 'task' not in request.json:
        return jsonify({"error": "Il campo 'task' è obbligatorio"}), 400
    todo['task'] = request.json['task']
    return jsonify(todo)

# DELETE /todos/<id> - Elimina un compito
@app.route('/todos/<int:todo_id>', methods=['DELETE'])
def delete_todo(todo_id):
    global todos
    todos = [todo for todo in todos if todo['id'] != todo_id]
    return '', 204

if __name__ == '__main__':
    app.run(debug=True)




 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat


# **Lezione 2.6 - Implementazione Json Client-Server**


In [ ]:
# Importazioni necessarie
from flask import Flask, request, jsonify
from werkzeug.exceptions import BadRequest, NotFound

app = Flask(__name__)

@app.route('/api/echo', methods=['POST'])
def echo():
    if not request.is_json:
        return jsonify({"error": "Request must be JSON"}), 415
    return jsonify(request.json)

@app.route('/api/process', methods=['POST'])
def process_data():
    data = request.get_json()
    if not data or 'operation' not in data or 'values' not in data:
        return jsonify({"error": "Invalid data format"}), 400

    operation = data['operation']
    values = data['values']

    if str(operation).lower() == 'sum':
        result = sum(values)
    elif str(operation).lower() == 'average':
        result = sum(values) / len(values) if values else 0
    else:
        return jsonify({"error": "Invalid operation"}), 400

    return jsonify({"result": result})

if __name__ == '__main__':
  app.run(debug=True)


# **Lezione 2.7 - Errore ed eccezioni con Flask**

In [ ]:
# Importazioni necessarie
from flask import Flask, request, jsonify
from werkzeug.exceptions import BadRequest, NotFound

app = Flask(__name__)

@app.errorhandler(404)
def not_found_error(error):
    return jsonify({"error": "Resource not found"}), 404

@app.errorhandler(400)
def bad_request_error(error):
    return jsonify({"error": "Bad request", "message": str(error)}), 400

@app.errorhandler(500)
def internal_server_error(error):
    return jsonify({"error": "Internal server error"}), 500

@app.route('/api/divide', methods=['POST'])
def divide():
    data = request.get_json()
    try:
        result = data['numerator'] / data['denominator']
        return jsonify({"result": result})
    except KeyError:
        raise BadRequest("Missing numerator or denominator")
    except ZeroDivisionError:
        return jsonify({"error": "Cannot divide by zero"}), 400
    except Exception as e:
        app.logger.error(f"Unexpected error: {str(e)}")
        return jsonify({"error": "An unexpected error occurred"}), 500

if __name__ == '__main__':
    app.run(debug=True)

# **Esercitazione**

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# Simulazione di un database
books = [
    {"id": 1, "title": "Il Nome della Rosa", "author": "Umberto Eco"},
    {"id": 2, "title": "1984", "author": "George Orwell"}
]

# Helper function per trovare un libro per ID
def find_book(book_id):
    return next((book for book in books if book['id'] == book_id), None)

# GET /books - Ottieni tutti i libri
@app.route('/books', methods=['GET'])
def get_books():
    return jsonify(books)

# GET /books/<id> - Ottieni un libro specifico
@app.route('/books/<int:book_id>', methods=['GET'])
def get_book(book_id):
    book = find_book(book_id)
    if book:
        return jsonify(book)
    return jsonify({"error": "Libro non trovato"}), 404

# POST /books - Aggiungi un nuovo libro
@app.route('/books', methods=['POST'])
def add_book():
    if not request.json or 'title' not in request.json or 'author' not in request.json:
        return jsonify({"error": "Dati mancanti"}), 400
    new_book = {
        'id': books[-1]['id'] + 1 if books else 1,
        'title': request.json['title'],
        'author': request.json['author']
    }
    books.append(new_book)
    return jsonify(new_book), 201

# PUT /books/<id> - Aggiorna un libro esistente
@app.route('/books/<int:book_id>', methods=['PUT'])
def update_book(book_id):
    book = find_book(book_id)
    if not book:
        return jsonify({"error": "Libro non trovato"}), 404
    if not request.json:
        return jsonify({"error": "Nessun dato fornito"}), 400
    book['title'] = request.json.get('title', book['title'])
    book['author'] = request.json.get('author', book['author'])
    return jsonify(book)

# DELETE /books/<id> - Elimina un libro
@app.route('/books/<int:book_id>', methods=['DELETE'])
def delete_book(book_id):
    book = find_book(book_id)
    if not book:
        return jsonify({"error": "Libro non trovato"}), 404
    books.remove(book)
    return jsonify({"message": "Libro eliminato con successo"}), 200

if __name__ == '__main__':
    app.run(debug=True)

Siamo una libreria e abbiamo al nostro interno un database di libri, composti dai seguenti parametri
- Id (autogenerato)
- Titolo
- Autore

Andiamo a creare una API Rest in flask che gestisca le CRUD per il database della nostra libreria.  
viene fornito un esempio del nostro database iniziale


```
# books = [
    {"id": 1, "title": "Il Nome della Rosa", "author": "Umberto Eco"},
    {"id": 2, "title": "1984", "author": "George Orwell"}
]
```
SUGGERIMENTO: Ricordarsi che le richieste vengono gestite dall'oggetto **request** e che l'oggetto **jsonify** ci aiuta a generare JSON per il return
